In [ ]:
%reload_ext autoreload
%autoreload 2

import time
import os
from datetime import datetime
import copy
from pathlib import Path
import json
from tqdm import tqdm
import sys
sys.path.append("..")

import torch
from torch_geometric.loader import DataLoader
from src.data.dataset import Crystals
from src.utils.loss import MSELoss, L1Loss, CosSimLoss, KLDivLoss, CombLoss
from src.utils.util import drop
import matplotlib.pyplot as plt
from src.models import MDNet

In [14]:
args = {
    "embedding_dimension": 64,
    "attn_activation": "silu",
    "num_heads": 8,
    "neighbor_embedding": True
}

device = torch.device('cuda')
model = MDNet(args).to(device)
# loss_fn = MSELoss()
loss_fn = CombLoss(
    (1, MSELoss()),
    (1, CosSimLoss())
    )
loss_name = repr(loss_fn)
model_name = str(model)
model_name = "TorchMD-net"
now = datetime.now()
dt_string = now.strftime("%Y%m%d-%H%M%S")
dataset_path = '../data/processed/v6.pt'
dataset = Crystals(dataset_path)
save_model_dir = Path(f'models/{model_name}_{loss_name}')
save_model_dir.mkdir(parents=True,exist_ok=True)

train_dataset, val_dataset, test_dataset = dataset.get_splits(deterministic=True)
train_loader = DataLoader(train_dataset, batch_size=64)
val_loader = DataLoader(val_dataset, batch_size=64)

fn = KLDivLoss()
def val(model, val_dataloader):
    model.eval()
    with torch.inference_mode():
        mse = 0
        for data in val_dataloader:
            data = data.to(device)
            pred = model(data)

            mse += fn(pred, data.y)
        return mse / len(val_dataloader)

train_loss = []
val_loss = []
display_epochs = 10
best_model_wts = copy.deepcopy(model.state_dict())
best_val_loss = 0.007
model_save_path = None

In [15]:
display_epochs = 1
def train(model,start,end,lr):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    global model_save_path, best_model_wts, best_val_loss
    try:
        for epoch in tqdm(range(start,end+1)):
            model.train()
            optimizer.zero_grad()
            for data in train_loader:
                data = data.to(device)
                pred = model(data)
                loss = loss_fn(pred, data.y)
                loss.backward()
                optimizer.step()

            train_kl = val(model, train_loader)
            val_kl = val(model, val_loader)
            train_loss.append(train_kl.item())
            val_loss.append(val_kl.item())

            if epoch % display_epochs == 0:
                tqdm.write(f"validation {loss_name} loss: {val_kl.item()}")
                tqdm.write(f"train {loss_name} loss: {train_kl.item()}")
            if val_kl < best_val_loss:
                best_val_loss = val_kl
                best_model_wts = copy.deepcopy(model.state_dict())
                model_save_path = save_model_dir / f'{dt_string}_epoch{epoch:05d}_kl{val_kl:.4f}.pt'
                torch.save(model.state_dict(),model_save_path)
                tqdm.write(f"validation {loss_name} loss: {val_kl.item()}")
                tqdm.write(f"model saved to {model_save_path}")
            time.sleep(3)
    except KeyboardInterrupt as e:
        Exception(e)
        tqdm.write(f"validation {loss_name} loss: {val_kl.item()}")
        tqdm.write(f"train {loss_name} loss: {train_kl.item()}")
        return epoch,val_kl,model_save_path,best_model_wts
    return epoch,val_kl,model_save_path,best_model_wts

In [ ]:
epoch,val_kl,model_save_path,best_model_wts = train(model,0,1000,lr=1e-4)

  0%|          | 0/1001 [00:05<?, ?it/s]

validation 1mse_1cossim loss: 0.05540403723716736
train 1mse_1cossim loss: 0.05587601289153099


  0%|          | 1/1001 [00:13<2:19:46,  8.39s/it]

validation 1mse_1cossim loss: nan
train 1mse_1cossim loss: nan


  0%|          | 1/1001 [00:15<4:23:27, 15.81s/it]

validation 1mse_1cossim loss: nan
train 1mse_1cossim loss: nan
